In [4]:
from Bio import SeqIO
import pandas as pd

hpv_seq_file = open(r"C:\Users\thaka\OneDrive\Desktop\CRISPR-cas12a test\GCF_001274345.1_ViralProj293723_genomic.fna","rt") 

sequences_obj_hpv = SeqIO.parse(hpv_seq_file, "fasta") 
sequences_hpv = [seq for seq in sequences_obj_hpv]

sequences_hpv_df = pd.DataFrame([{"id": s.id, "seq": str(s.seq)} for s in sequences_hpv]) 

print(f"Starting Base pairs {sequences_hpv[0].seq[:20]}")

# AT_content = sum(seq.seq.count('A') + seq.seq.count('T') for seq in sequences_hpv) / sum(len(seq.seq) for seq in sequences_hpv) #general formula to calculate AT content of a sequence. It counts the number of A's and T's in the sequence and divides it by the total length of the sequence.
# print(f"AT content of HPV sequence : {AT_content * 100:.3f}%")

from sklearn.feature_extraction.text import CountVectorizer

def to_kmers(seq, k=4):
    return ' '.join(seq[i:i+k] for i in range(len(seq) - k + 1))

kmer_text = sequences_hpv_df['seq'].apply(to_kmers)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(kmer_text)



Starting Base pairs ATGGCCACTGAGCATCCAAG


In [5]:
from Bio import SeqIO
import pandas as pd

hiv_seq_file = open(r"C:\Users\thaka\OneDrive\Desktop\CRISPR-cas12a test\GCA_000864765.1_ViralProj15476_genomic.fna","rt") 

sequences_obj_hiv = SeqIO.parse(hiv_seq_file, "fasta") 
sequences_hiv = [seq for seq in sequences_obj_hiv]

sequences_hiv_df = pd.DataFrame([{"id": s.id, "seq": str(s.seq)} for s in sequences_hiv]) 

print(f"Starting Base pairs {sequences_hiv[0].seq[:20]}")

# AT_content = sum(seq.seq.count('A') + seq.seq.count('T') for seq in sequences_hiv) / sum(len(seq.seq) for seq in sequences_hiv) #general formula to calculate AT content of a sequence. It counts the number of A's and T's in the sequence and divides it by the total length of the sequence.
# print(f"AT content of HIV sequence : {AT_content * 100:.3f}%")

from sklearn.feature_extraction.text import CountVectorizer

def to_kmers(seq, k=4):
    return ' '.join(seq[i:i+k] for i in range(len(seq) - k + 1))

kmer_text = sequences_hiv_df['seq'].apply(to_kmers)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(kmer_text)


Starting Base pairs GGTCTCTCTGGTTAGACCAG


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

def chunk(seq, size_bp=200): return [seq[i:i+size_bp] for i in range(0, len(seq) - size_bp + 1, size_bp)]

frags = pd.DataFrame([(f, "HPV") for s in sequences_hpv_df["seq"] for f in chunk(s)] + [(f, "HIV") for s in sequences_hiv_df["seq"] for f in chunk(s)], columns=["seq", "label"])

# every fragment becomes one row of 4-mer counts, so X has one row per fragment and 256 columns
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(frags["seq"].apply(to_kmers))
labels = frags["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels
)

print(X_train.shape[0], "to study")
print(X_test.shape[0], "to be tested on...")

64 to study
17 to be tested on...


In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
print("accuracy:", model.score(X_test, y_test))

accuracy: 0.8235294117647058
